# as-strided-windowing composite — cx4: conv via as_strided windowing + explicit sum-reduce over C_in

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `as-strided-windowing`, `conv-channel-sum`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "as-strided-windowing"
DD_ATOM_IDS = ["as-strided-windowing", "conv-channel-sum"]
DD_SUBTOPICS = ["PyTorch: as_strided windowing", "CNN: Channel-axis sum semantics"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Most Conv2d explanations focus on the spatial axes, but the C_in axis matters too:

- **conv-channel-sum** — every output channel `o` is `sum over c_in of (input_channel_c * filter_c)`. That "sum over c_in" is what makes Conv2d a TENSOR operation rather than a stack of per-channel 2-D convs.
- **as-strided-windowing** — gives you a `(B, C_in, H_out, W_out, K, K)` patch view. The C_in axis is already there — you just need to reduce it.

The composition: with the patch view in hand, the conv arithmetic for ONE output channel is `(patches * w[o]).sum(dim=(1, 4, 5))` — multiply broadcasts the C_in / K / K axes, then an explicit `.sum(dim=...)` collapses them. The C_in axis is the one we care about here — it's the canonical channel-mixing reduction.

Why an explicit sum and not einsum? **Auditability**. When you're debugging "why are my filters producing all-zeros after init", a per-axis `.sum` is easier to break into intermediate prints than an einsum string. ARENA's `conv2d_minimal` uses einsum; ARENA's debugging exercises sometimes use the explicit form.

### Composite Exercise — conv via as_strided windowing + explicit sum-reduce over C_in

**Atoms exercised together**: `as-strided-windowing`, `conv-channel-sum`

Implement `cx4_conv2d_via_sum(x, w)`.

- `x`: float tensor of shape `(B, C_in, H, W)`. Stride=1, pad=0.
- `w`: float tensor of shape `(C_out, C_in, K, K)`.

Return a float tensor of shape `(B, C_out, H_out, W_out)` matching `F.conv2d(x, w)`.

1. **as-strided-windowing** — build a 6-D view of shape `(B, C_in, H_out, W_out, K, K)`.
2. **conv-channel-sum** — for each output channel:
   - broadcast `patches` against `w[o]` (insert a leading 1 in `w` and a singleton C_out-style dim — easiest: process one `o` at a time in Python, then stack), OR
   - do it in one go: `(patches.unsqueeze(1) * w.view(1, C_out, C_in, 1, 1, K, K)).sum(dim=(2, -2, -1))` — multiply elementwise against the broadcast weight, then explicitly `.sum` over `C_in`, `KH`, `KW` axes.

The test asserts the result agrees with `F.conv2d` AND probes the per-channel sum decomposition (by checking that zeroing one C_in axis of `w` zeros the corresponding contribution).

In [ ]:
def cx4_conv2d_via_sum(x, w):
    B, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    H_out = H - KH + 1
    W_out = W - KW + 1
    # Atom A (as-strided-windowing): (B, C_in, H_out, W_out, KH, KW) patch view.
    sB, sC, sH, sW = x.stride()
    patches = x.as_strided(
        size=(B, C_in, H_out, W_out, KH, KW),
        stride=(sB, sC, sH, sW, sH, sW),
    )
    # Atom B (conv-channel-sum): explicit reduce over C_in, KH, KW.
    # Broadcast shapes:
    #   patches: (B, 1,     C_in, H_out, W_out, KH, KW)
    #   w:       (1, C_out, C_in, 1,     1,     KH, KW)
    p = patches.unsqueeze(1)
    wb = w.view(1, C_out, C_in, 1, 1, KH, KW)
    prod = p * wb  # (B, C_out, C_in, H_out, W_out, KH, KW)
    return prod.sum(dim=(2, -2, -1))  # sum over C_in, KH, KW -> (B, C_out, H_out, W_out)


<details><summary>Show solution — cx4</summary>

```python
def cx4_conv2d_via_sum(x, w):
    B, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    H_out = H - KH + 1
    W_out = W - KW + 1
    # Atom A (as-strided-windowing): (B, C_in, H_out, W_out, KH, KW) patch view.
    sB, sC, sH, sW = x.stride()
    patches = x.as_strided(
        size=(B, C_in, H_out, W_out, KH, KW),
        stride=(sB, sC, sH, sW, sH, sW),
    )
    # Atom B (conv-channel-sum): explicit reduce over C_in, KH, KW.
    # Broadcast shapes:
    #   patches: (B, 1,     C_in, H_out, W_out, KH, KW)
    #   w:       (1, C_out, C_in, 1,     1,     KH, KW)
    p = patches.unsqueeze(1)
    wb = w.view(1, C_out, C_in, 1, 1, KH, KW)
    prod = p * wb  # (B, C_out, C_in, H_out, W_out, KH, KW)
    return prod.sum(dim=(2, -2, -1))  # sum over C_in, KH, KW -> (B, C_out, H_out, W_out)
```

The `.sum(dim=(2, -2, -1))` is the channel-sum atom made explicit: `dim=2` is C_in (after the broadcast unsqueeze), `dim=-2, -1` are KH, KW. This is the explicit-reduce form of the same operation einsum would do — slower but more debuggable. The Case-B sanity (zero one C_in row of w, expect that channel's contribution to disappear) is the canonical test of the channel-sum atom.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx4',
        'subtopics': ["PyTorch: as_strided windowing", "CNN: Channel-axis sum semantics"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()